# **基于 YOLOv11 的疲劳驾驶检测**

这个 notebook 使用 **YOLOv11** 直接训练本地分类数据集。内容包括环境检查、本地数据集读取、图片预览、YOLOv11 分类训练、验证集预测、结果可视化、模型备份，以及 TensorFlow Lite 导出。

目标类别是 `drowsy` 和 `notdrowsy`。


## 使用 YOLOv11 做实时疲劳驾驶检测

本节使用 **YOLOv11** 进行疲劳驾驶分类检测。**YOLO (You Only Look Once)** 适合实时视觉任务，可以在较短推理时间内完成 `drowsy` 和 `notdrowsy` 分类。

本项目直接使用本地已经整理好的分类数据集，默认数据集路径为 `/home/hebu/dd_dataset`。


## 配置 YOLOv11 环境

本节安装运行 YOLOv11 所需的依赖。

### 安装依赖

需要安装以下包：

- **Ultralytics**：YOLOv11 的官方工具库，用于训练、预测和导出模型。
- **TensorFlow**：用于后续导出 TensorFlow Lite 模型。
- **matplotlib / pillow / opencv-python**：用于图片读取、展示和结果图查看。


In [ ]:
!pip install ultralytics==8.3.40
!pip install tensorflow==2.13.1
!pip install matplotlib pillow opencv-python


### 检查安装

依赖安装完成后，运行检查命令确认 Ultralytics 环境是否可用，避免后续训练时才发现配置问题。


In [ ]:
from ultralytics import YOLO
from IPython.display import display , Image
from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()

## 加载本地数据集

本项目直接使用本地数据集，默认路径为 `/home/hebu/dd_dataset`。

YOLO 分类训练推荐的数据结构如下：

```text
/home/hebu/dd_dataset/
  train/
    drowsy/
    notdrowsy/
  valid/
    drowsy/
    notdrowsy/
  test/        # 可选
    drowsy/
    notdrowsy/
```

如果你的验证集文件夹叫 `val` 而不是 `valid`，下面代码也会自动识别。


In [ ]:
from pathlib import Path
import os

# 本地数据集路径
DATASET_DIR = Path("/home/hebu/dd_dataset")
dataset_dir = str(DATASET_DIR)

if not DATASET_DIR.exists():
    raise FileNotFoundError(f"未找到数据集目录: {DATASET_DIR}，请确认路径是否正确。")

train_dir = DATASET_DIR / "train"
valid_dir = DATASET_DIR / "valid"
if not valid_dir.exists():
    valid_dir = DATASET_DIR / "val"

if not train_dir.exists():
    raise FileNotFoundError(f"未找到训练集目录: {train_dir}")
if not valid_dir.exists():
    raise FileNotFoundError("未找到验证集目录，请确认存在 /home/hebu/dd_dataset/valid 或 /home/hebu/dd_dataset/val。")

print(f"数据集路径: {DATASET_DIR}")
print(f"训练集路径: {train_dir}")
print(f"验证集路径: {valid_dir}")

# 打印目录结构，方便确认类别文件夹
for root, dirs, files in os.walk(DATASET_DIR):
    level = Path(root).relative_to(DATASET_DIR).parts
    if len(level) > 2:
        continue
    print(f"当前目录: {root}")
    print(f"子目录: {dirs}")
    print(f"文件数量: {len(files)}")
    print("-" * 40)


## 展示数据集图片

本节从训练集中抽取一部分图片，并用 `matplotlib` 可视化。

代码使用 `glob` 递归读取图片路径，再选择其中一小段样本进行展示。随后用 `PIL` 打开图片，并按照 3x4 网格显示。

如果某张图片损坏或无法读取，代码会捕获异常，并在对应子图中显示加载失败提示。这个步骤主要用于快速检查数据是否正常、类别图片是否符合预期。


In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

# 读取训练集图片路径，可按需要调整切片范围
image_paths = glob.glob(f"{dataset_dir}/train/**/*.*", recursive=True)[24:36]

# 检查图片数量是否足够展示
if len(image_paths) < 12:
    raise ValueError(f"图片数量不足，只找到 {len(image_paths)} 张图片。")

# 创建 3x4 子图
fig, axes = plt.subplots(3, 4, figsize=(20, 15))

# 遍历图片并显示到子图中
for i, image_path in enumerate(image_paths):
    # 计算当前图片所在的行和列
    row = i // 4
    col = i % 4

    try:
        # 使用 PIL 打开图片
        img = Image.open(image_path)

        # 在当前子图中显示图片
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        axes[row, col].set_title(image_path.split('/')[-1])
    except Exception as e:
        print(f"显示图片失败 {image_path}: {e}")
        axes[row, col].axis('off')
        axes[row, col].set_title("图片加载失败")

# 调整子图间距
plt.tight_layout()

# 显示图片
plt.show()


## 训练 YOLOv11 分类模型

本节使用预训练的 YOLOv11 分类模型 `yolo11n-cls.pt` 作为起点，在本地数据集 `/home/hebu/dd_dataset` 上继续训练。预训练模型已经学习过通用图像特征，因此可以减少训练时间和资源消耗。

### 训练逻辑

代码会优先检查本地是否已有训练好的 `best.pt`。如果存在，就直接加载；如果没有，则检查是否有 `runs_backup.zip` 可恢复；如果两者都没有，就从 `yolo11n-cls.pt` 开始训练。

训练时使用 `model.train(data=dataset_dir, epochs=20)`。这里设置 20 个 epoch，适合快速实验；如果要追求更稳定的效果，可以根据数据规模和硬件条件调整 epoch 数。


In [ ]:
import os
import zipfile
from ultralytics import YOLO

# 路径配置
runs_directory = "runs/classify/train/weights"
best_model_path = os.path.join(runs_directory, "best.pt")
backup_zip = "runs_backup.zip"

# 如果已经存在训练好的模型，直接加载
if os.path.exists(best_model_path):
    print(f"找到已训练模型 '{best_model_path}'，正在加载 YOLO...")
    model = YOLO(best_model_path)

# 如果没有模型，则检查是否存在备份压缩包
elif os.path.exists(backup_zip):
    print(f"找到 '{backup_zip}'，正在解压到 '{runs_directory}'...")
    with zipfile.ZipFile(backup_zip, 'r') as zip_ref:
        zip_ref.extractall("runs")
    print("解压完成，正在加载模型...")

    if os.path.exists(best_model_path):
        model = YOLO(best_model_path)
    else:
        print("解压后未找到 best.pt，需要重新训练。")
        model = YOLO("yolo11n-cls.pt")
        model.train(data=dataset_dir, epochs=20)
        print("训练完成。")

# 如果既没有模型也没有备份，则从头训练
else:
    print("未找到已训练模型或备份，开始训练...")
    model = YOLO("yolo11n-cls.pt")
    model.train(data=dataset_dir, epochs=20)
    print("训练完成。")

    # 训练完成后备份 runs 目录
    print("正在备份训练结果...")
    with zipfile.ZipFile(backup_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk("runs"):
            for file in files:
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.relpath(file_path, "runs"))
    print(f"备份已保存为 '{backup_zip}'")


### 在验证集图片上进行分类预测

本节自动读取验证集下的类别文件夹，并从每个类别中抽样图片进行预测。当前数据集的类别是 `drowsy` 和 `notdrowsy`。

流程如下：

1. 设置验证集路径：优先使用 `/home/hebu/dd_dataset/valid`，如果不存在则使用 `/home/hebu/dd_dataset/val`。
2. 自动读取类别文件夹：不手动写死类别名。
3. 收集图片文件：读取常见图片格式。
4. 随机抽样：每个类别最多抽取 100 张。
5. 模型预测：对每张图片运行 `model(image_path)`。
6. 取最高概率类别：将概率最高的类别作为预测结果。
7. 计算准确率：统计预测正确数量，并输出最终得分。


In [ ]:
import random
from pathlib import Path

# 验证集图片文件夹
valid_images_path = Path(valid_dir)
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# 自动读取验证集下的类别文件夹
subfolders = sorted([p.name for p in valid_images_path.iterdir() if p.is_dir()])
print(f"检测到类别: {subfolders}")

# 每个类别最多抽样的图片数量
sample_size = 100

# 从每个类别文件夹中递归收集抽样图片路径
image_paths = []
true_labels = []
for subfolder in subfolders:
    folder_path = valid_images_path / subfolder
    all_images = [p for p in folder_path.rglob('*') if p.is_file() and p.suffix.lower() in image_exts]
    print(f"{subfolder}: 找到 {len(all_images)} 张验证图片")

    # 如果图片不足 sample_size，则使用该类别下全部图片
    sampled_images = random.sample(all_images, min(sample_size, len(all_images)))

    for img in sampled_images:
        image_paths.append(str(img))
        true_labels.append(subfolder)

# 评估计数器
correct_predictions = 0
total_images = len(image_paths)

# 遍历抽样图片并进行预测
for image_path, true_label in zip(image_paths, true_labels):
    results = model(image_path)

    for result in results:
        # 将概率转换为普通列表
        probs = result.probs.data.tolist()
        classes = result.names

        highest_prob = max(probs)
        highest_prob_index = probs.index(highest_prob)
        predicted_class = classes[highest_prob_index]

        # 判断预测类别是否与真实类别一致
        is_correct = "正确" if predicted_class == true_label else "错误"
        if predicted_class == true_label:
            correct_predictions += 1

        print(f"图片: {image_path} - 预测: {predicted_class} ({highest_prob*100:.2f}%) - 真实: {true_label} -> {is_correct}")

# 输出最终准确率
accuracy = (correct_predictions / total_images) * 100 if total_images > 0 else 0
print(f"\n最终得分: {correct_predictions}/{total_images} 预测正确，准确率 {accuracy:.2f}%")


这个过程会遍历抽样到的验证集图片，逐张输出预测类别、置信度和真实标签，便于观察模型在不同样本上的表现。


### 模型评估与混淆矩阵

- **Confusion Matrix**：保存在 `runs/classify/train` 目录中，用于查看模型对每个类别的分类效果。
- **归一化混淆矩阵**：更方便比较不同类别的相对表现。
- **训练结果图**：展示训练过程中 accuracy、loss 等指标变化。

在 **Google Colab** 中，可以使用 `cv2_imshow()` 直接显示这些图片。


In [ ]:
from google.colab.patches import cv2_imshow
import cv2

confusion_matrix_path = "./runs/classify/train/confusion_matrix.png"

# 读取并显示混淆矩阵图片
cv2_imshow(cv2.imread(confusion_matrix_path))


In [ ]:
confusion_matrix_normalized_path = "./runs/classify/train/confusion_matrix_normalized.png"

# 读取并显示归一化混淆矩阵图片
cv2_imshow(cv2.imread(confusion_matrix_normalized_path))


In [ ]:
results_path = "./runs/classify/train/results.png"

# 读取并显示训练结果图片
cv2_imshow(cv2.imread(results_path))


In [ ]:
import shutil
import os

# 需要压缩的目录
runs_directory = "./runs"
zip_filename = "runs_backup.zip"

# 如果 runs 目录存在，则创建 ZIP 备份
if os.path.exists(runs_directory):
    shutil.make_archive(zip_filename.replace(".zip", ""), 'zip', runs_directory)
    print(f"压缩完成: {zip_filename}")
else:
    print("runs 目录不存在，无需压缩。")


### 保存和备份 YOLOv11 模型

训练完成后，模型权重会保存在 `runs/classify/train/weights` 中，其中 `best.pt` 是验证效果较好的权重文件。

为了便于复用，notebook 会将 `runs` 目录压缩为 `runs_backup.zip`。后续重新运行时，如果检测到已有 `best.pt` 或备份文件，就可以直接加载或恢复，不必每次重新训练。


#### 本地部署说明

训练完成后，主要使用的模型文件是：

```text
runs/classify/train/weights/best.pt
```

你可以直接用 Ultralytics 的 `YOLO(best_model_path)` 加载它进行本地推理，也可以继续导出为 TensorFlow Lite，用于移动端或嵌入式设备。


## 将 YOLOv11 权重导出为 TensorFlow Lite

如果要在移动端部署，需要将训练好的 `best.pt` 权重转换为 TensorFlow Lite 格式。TensorFlow Lite 模型体积更小，推理更适合移动设备和嵌入式设备。


In [ ]:
# 将模型导出为 TensorFlow Lite 格式
model.export(format="tflite")

print("模型已成功转换为 TensorFlow Lite。")
